In [1]:
import pandas as pd
from readback_guard.config import load_config
from readback_guard.evaluate import load_cases, gold_clearance, rates, log_result
from readback_guard.extract import extract

cfg = load_config()
dev = load_cases(cfg, "dev")
dev.groupby(["kind", "verdict"]).size()

kind      verdict       
injected  WRONG_CALLSIGN    12
          WRONG_VALUE       23
real      CORRECT           37
          MISSING_ITEM       8
          WRONG_CALLSIGN     3
          WRONG_VALUE        4
dtype: int64

In [2]:
row = dev.iloc[0]
print(row.atco_text)
extract(row.atco_text, cfg)

hotel zulu mike sierra four runway three one cleared for takeoff wind one eight zero degrees seven knots good bye


Clearance(callsign='hzms4', items=[Item(type='runway', value='31', unit=None, qualifier=None)])

In [2]:
texts = pd.unique(pd.concat([dev.atco_text, dev.pilot_text]))
ext = {}
for n, text in enumerate(texts, 1):
    ext[text] = extract(text, cfg)
    if n % 20 == 0:
        print(n, "/", len(texts))
len(ext)

20 / 139
40 / 139
60 / 139
80 / 139
100 / 139
120 / 139


139

In [3]:
from readback_guard.compare import compare

dev["pred"] = [compare(gold_clearance(r.atco_callsign, r.atco_items),
                       gold_clearance(r.pilot_callsign, r.pilot_items)).verdict
               for r in dev.itertuples()]
m_oracle = rates(dev)
m_oracle

{'n_errors': 50,
 'detected': 50,
 'detection_rate': 1.0,
 'detection_ci': (0.929, 1.0),
 'n_correct': 37,
 'false_alarms': 0,
 'false_alarm_rate': 0.0,
 'false_alarm_ci': (0.0, 0.094),
 'exact_verdict_rate': 0.989}

In [4]:
verdicts = [compare(ext[r.atco_text], ext[r.pilot_text]) for r in dev.itertuples()]
dev["pred"] = [v.verdict for v in verdicts]
dev["reason"] = [v.reason for v in verdicts]
m_text = rates(dev)
m_text

{'n_errors': 50,
 'detected': 47,
 'detection_rate': 0.94,
 'detection_ci': (0.838, 0.979),
 'n_correct': 37,
 'false_alarms': 2,
 'false_alarm_rate': 0.054,
 'false_alarm_ci': (0.015, 0.177),
 'exact_verdict_rate': 0.874}

In [5]:
print(pd.crosstab(dev.verdict, dev.pred))
dev.loc[dev.pred != dev.verdict, ["case_id", "atco_text", "pilot_text", "verdict", "pred", "reason"]]

pred            CORRECT  MISSING_ITEM  WRONG_CALLSIGN  WRONG_VALUE
verdict                                                           
CORRECT              35             0               1            1
MISSING_ITEM          1             4               1            2
WRONG_CALLSIGN        1             0              14            0
WRONG_VALUE           1             1               2           23


,case_id,atco_text,pilot_text,verdict,pred,reason
1,p0749,csa five three six runway one three cleared fo...,go runway one three csa five three si,CORRECT,WRONG_CALLSIGN,callsign csa 536 read back as csa 53s
5,p0664,easy four nine six papa roger use romeo and st...,hold romeo for easy four nine six pa,MISSING_ITEM,WRONG_CALLSIGN,callsign easy496pr read back as easy496pa
11,p0440,csa nine seven five runway one three cleared t...,cleared to land csa nine seven five,MISSING_ITEM,WRONG_VALUE,runway 13 read back as 31
27,p0750,csa five alfa juliett line up runway one three,up runway one three csa five alfa juliett,CORRECT,WRONG_VALUE,runway 13 read back as 13C
31,p0178,csa zero two four line up runway three one,line up runway three,MISSING_ITEM,WRONG_VALUE,runway 31 read back as 03
38,p0526,csa four eight one contact ruzyne ground one t...,one decimal nine csa four eight one,MISSING_ITEM,CORRECT,all items read back
49,p0264,three zero three reduce speed one five zero kn...,speed one five or less csa three zero three,WRONG_VALUE,CORRECT,all items read back
52,p0749_inj,csa five three six runway one three cleared fo...,go runway one one csa five three si,WRONG_VALUE,WRONG_CALLSIGN,callsign csa 536 read back as csa 53s
69,p0034_inj,air france one three two runway three one clea...,three three air france one three two,WRONG_VALUE,MISSING_ITEM,runway 31 not read back
76,p0511_inj,csa seven zero two runway one three cleared fo...,runway one three cleared for seven zero three,WRONG_CALLSIGN,CORRECT,all items read back


In [6]:
from readback_guard.normalize import norm_callsign, norm_value
from readback_guard.schemas import SCORED_TYPES

def scored(clr):
    return {(i.type, norm_value(i.type, i.value, i.unit)) for i in clr.items if i.type in SCORED_TYPES}

rows = []
for r in dev.itertuples():
    for side in ("atco", "pilot"):
        text = getattr(r, f"{side}_text")
        gold = gold_clearance(getattr(r, f"{side}_callsign"), getattr(r, f"{side}_items"))
        rows.append({"side": side, "text": text,
                     "callsign_ok": norm_callsign(ext[text].callsign) == norm_callsign(gold.callsign),
                     "items_ok": scored(ext[text]) == scored(gold)})
acc = pd.DataFrame(rows).drop_duplicates(["side", "text"])
acc.groupby("side")[["callsign_ok", "items_ok"]].mean()

,callsign_ok,items_ok
side,,
atco,0.942308,0.961538
pilot,0.850575,0.885057


In [7]:
log_result("exp1a_gold_fields", "dev", "gold", m_oracle, cfg)
log_result("exp1b_text_pipeline", "dev", "ref", m_text, cfg, notes="prompt v1")

In [8]:
from typing import TypedDict
from readback_guard.schemas import Clearance, Verdict

class State(TypedDict, total=False):
    atco_text: str
    pilot_text: str
    atco: Clearance
    pilot: Clearance
    verdict: Verdict

In [9]:
def extract_node(state):
    return {"atco": extract(state["atco_text"], cfg), "pilot": extract(state["pilot_text"], cfg)}

def compare_node(state):
    return {"verdict": compare(state["atco"], state["pilot"])}

def gate_node(state):
    return {}

In [10]:
from google.genai import errors
from langgraph.graph import END, START, StateGraph
from langgraph.types import RetryPolicy
from pydantic import ValidationError

retry = RetryPolicy(max_attempts=3, retry_on=(ValidationError, errors.ServerError))
g = StateGraph(State)
g.add_node("extract", extract_node, retry_policy=retry)
g.add_node("compare", compare_node)
g.add_node("gate", gate_node)
g.add_edge(START, "extract")
g.add_edge("extract", "compare")
g.add_edge("compare", "gate")
g.add_edge("gate", END)
app = g.compile()

out = app.invoke({"atco_text": dev.atco_text[0], "pilot_text": dev.pilot_text[0]})
out["verdict"]

Verdict(verdict='MISSING_ITEM', reason='runway 31 not read back', items=[ItemResult(type='runway', atco_value='31', pilot_value=None, result='MISSING')])

In [11]:
print(app.get_graph().draw_mermaid())
same = sum(app.invoke({"atco_text": r.atco_text, "pilot_text": r.pilot_text})["verdict"].verdict == r.pred
           for r in dev.itertuples())
same, len(dev)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract(extract)
	compare(compare)
	gate(gate)
	__end__([<p>__end__</p>]):::last
	__start__ --> extract;
	compare --> gate;
	extract --> compare;
	gate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



(87, 87)